In [ ]:
from pathlib import Path

from sklearn.model_selection import train_test_split

# Obtener todos los archivos .C en el directorio 
c_files = list(Path("soco-dataset/c").glob("*.c"))
java_files = list(Path("soco-dataset/java").glob("*.java"))

files = c_files + java_files

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Utilizar tokenizer y modelo de embedding UniXcoder
tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")
embedding_model = AutoModel.from_pretrained("microsoft/unixcoder-base")

print("Successfully imported tokenizer and model UNIXCODER")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Successfully imported tokenizer and model UNIXCODER


In [ ]:
def tokenize_and_embed(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        code = f.read()

    tokens = tokenizer(
        code,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = embedding_model(**tokens)

    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

def save_embeddings(files):
    embeddings = []
    
    for file in tqdm(
        files,
        desc="🎀 KATSEYE EMBEDDINGS 🎀",
        unit="file"
    ):
        embedded_code = tokenize_and_embed(file)
        embeddings.append(embedded_code)

    embeddings = np.array(embeddings)
    embeddings = embeddings.squeeze(1)
    
    return embeddings

In [ ]:
embeddings = save_embeddings(files)

🎀 KATSEYE EMBEDDINGS 🎀:   0%|          | 0/338 [00:00<?, ?file/s]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings)

In [ ]:
""" import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

plt.imshow(sim_matrix, cmap="RdPu")
plt.colorbar(label="Cosine Similarity")

plt.title("Similarity Matrix")
plt.xlabel("File Index")
plt.ylabel("File Index")

plt.show() """

' import matplotlib.pyplot as plt\n\nplt.figure(figsize=(10, 8))\n\nplt.imshow(sim_matrix, cmap="RdPu")\nplt.colorbar(label="Cosine Similarity")\n\nplt.title("Similarity Matrix")\nplt.xlabel("File Index")\nplt.ylabel("File Index")\n\nplt.show() '

In [ ]:
""" threshold = 0.95

rows, cols = np.where(sim_matrix > threshold)

plt.figure(figsize=(10, 10))
plt.scatter(cols, rows, s=10)
plt.xlabel("File")
plt.ylabel("File")
plt.title("Similar Pairs (>0.95)")
plt.show() """

' threshold = 0.95\n\nrows, cols = np.where(sim_matrix > threshold)\n\nplt.figure(figsize=(10, 10))\nplt.scatter(cols, rows, s=10)\nplt.xlabel("File")\nplt.ylabel("File")\nplt.title("Similar Pairs (>0.95)")\nplt.show() '

In [ ]:
def sort_similarities(files, embeddings, sim_matrix):
    pairs = []
    
    for i in range(len(embeddings)):
        for j in range(i + 1, len(embeddings)):
            plagiarism = 0
            if sim_matrix[i][j] > 0.8:
                plagiarism = 1
            pairs.append((files[i], files[j], round(float(sim_matrix[i][j]), 4), plagiarism))
    
    pairs.sort(key=lambda x: x[2], reverse=True)

    return pairs

In [ ]:
def create_dataframe(pairs, num_of_pairs):
    data_frame = pd.DataFrame(
    
        pairs[:num_of_pairs],
    
        columns=["file1", "file2", "similarity", "plagiarism_suspected"]
    
    )
    
    data_frame["file1"] = data_frame["file1"].apply(lambda x: x.name)
    data_frame["file2"] = data_frame["file2"].apply(lambda x: x.name)

    return data_frame
    

In [ ]:
pairs = sort_similarities(files, embeddings, sim_matrix)

dataframe = create_dataframe(pairs, 60000)

display(dataframe.head(100))

,file1,file2,similarity,plagiarism_suspected
0,035.c,036.c,1.0000,1
1,005.java,006.java,0.9999,1
2,015.java,023.java,0.9990,1
3,043.java,251.java,0.9932,1
4,024.java,016.java,0.9927,1
...,...,...,...,...
95,124.java,077.java,0.9088,1
96,010.c,043.c,0.9087,1
97,022.c,042.c,0.9084,1
98,009.c,051.c,0.9082,1


In [ ]:
def extract_features(code):
    return {
        "num_loops": count_loops(code),
        "num_ifs": count_ifs(code),
        "num_functions": count_functions(code),
        "num_variables": count_variables(code),
        "avg_function_length": avg_function_length(code)
    }

# Dataset Construction: Similarity Features

Usamos los archivos `.qrel` como ground truth real (en vez del threshold de coseno) y calculamos las métricas de similitud textual para cada par.

In [ ]:

import subprocess
subprocess.run(["pip", "install", "rapidfuzz", "-q"], capture_output=True)

import re
import math
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path
from rapidfuzz.distance import Levenshtein as RapidLev
from rapidfuzz.distance import Jaro, JaroWinkler

# Cargar ground truth desde archivos .qrel
def load_ground_truth():
    plagiarism_pairs = set()
    for qrel_path in ["soco-dataset/SOCO14-c.qrel", "soco-dataset/SOCO14-java.qrel"]:
        with open(qrel_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 2:
                    a, b = parts
                    plagiarism_pairs.add((a, b))
                    plagiarism_pairs.add((b, a))
    return plagiarism_pairs

ground_truth = load_ground_truth()

print(ground_truth)

print(f"Pares de plagio en ground truth: {len(ground_truth) // 2}")


{('258.java', '059.java'), ('145.java', '143.java'), ('087.java', '155.java'), ('183.java', '258.java'), ('106.java', '111.java'), ('007.c', '043.c'), ('105.java', '103.java'), ('064.c', '026.c'), ('117.java', '119.java'), ('183.java', '051.java'), ('232.java', '233.java'), ('222.java', '155.java'), ('246.java', '244.java'), ('051.java', '257.java'), ('146.java', '147.java'), ('222.java', '087.java'), ('222.java', '153.java'), ('148.java', '150.java'), ('042.c', '045.c'), ('133.java', '131.java'), ('112.java', '108.java'), ('078.java', '079.java'), ('026.c', '064.c'), ('153.java', '222.java'), ('183.java', '059.java'), ('235.java', '237.java'), ('185.java', '258.java'), ('242.java', '087.java'), ('059.java', '258.java'), ('061.java', '216.java'), ('005.c', '006.c'), ('004.java', '003.java'), ('113.java', '108.java'), ('071.c', '059.c'), ('086.java', '222.java'), ('119.java', '117.java'), ('201.java', '209.java'), ('257.java', '048.java'), ('056.c', '057.c'), ('064.java', '062.java'), (

In [ ]:

def read_code(filename):
    for folder in ["soco-dataset/c", "soco-dataset/java"]:
        path = Path(folder) / filename
        if path.exists():
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
    return ""

def normalize_code(code):
    # Eliminar comentarios de línea y bloque
    code = re.sub(r"//.*?\n|/\*.*?\*/", " ", code, flags=re.DOTALL)
    return re.sub(r"\s+", " ", code).strip().lower()

def tokenize(code):
    return re.findall(r"\w+", code)

def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

def jaccard(seq_a, seq_b):
    set_a, set_b = set(seq_a), set(seq_b)
    if not set_a and not set_b:
        return 1.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

def dice(seq_a, seq_b):
    set_a, set_b = set(seq_a), set(seq_b)
    if not set_a and not set_b:
        return 1.0
    inter = len(set_a & set_b)
    denom = len(set_a) + len(set_b)
    return 2 * inter / denom if denom > 0 else 0.0

def cosine_bow(tokens_a, tokens_b):
    ca, cb = Counter(tokens_a), Counter(tokens_b)
    vocab = set(ca) | set(cb)
    dot = sum(ca[w] * cb[w] for w in vocab)
    mag_a = math.sqrt(sum(v ** 2 for v in ca.values()))
    mag_b = math.sqrt(sum(v ** 2 for v in cb.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

def compute_all_features(file1, file2):
    code1 = normalize_code(read_code(file1))
    code2 = normalize_code(read_code(file2))
    tokens1 = tokenize(code1)
    tokens2 = tokenize(code2)

    # Limitar tokens para que Levenshtein/Jaro sean razonablemente rápidos
    t1 = " ".join(tokens1[:300])
    t2 = " ".join(tokens2[:300])

    lev_dist = RapidLev.distance(t1, t2)
    max_len = max(len(t1), len(t2))
    lev_ratio = 1.0 - (lev_dist / max_len) if max_len > 0 else 1.0

    return {
        "jaccard_tokens":    jaccard(tokens1, tokens2),
        "jaccard_bigrams":   jaccard(get_ngrams(tokens1, 2), get_ngrams(tokens2, 2)),
        "jaccard_trigrams":  jaccard(get_ngrams(tokens1, 3), get_ngrams(tokens2, 3)),
        "dice":              dice(tokens1, tokens2),
        "levenshtein_dist":  lev_dist,
        "levenshtein_ratio": lev_ratio,
        "jaro":              Jaro.similarity(t1, t2),
        "jaro_winkler":      JaroWinkler.similarity(t1, t2),
        "cosine_similarity": cosine_bow(tokens1, tokens2),
    }

print("Funciones de similitud definidas correctamente.")


Funciones de similitud definidas correctamente.


In [ ]:

from tqdm.notebook import tqdm

# Etiquetar TODOS los pares con ground truth real
df_labeled = dataframe.copy()
df_labeled["plagiarism"] = df_labeled.apply(
    lambda row: 1 if (row["file1"], row["file2"]) in ground_truth else 0,
    axis=1
)

print(f"Total pares disponibles: {len(df_labeled)}")
print(f"  Plagio    (label=1): {df_labeled['plagiarism'].sum()}")
print(f"  No plagio (label=0): {(df_labeled['plagiarism'] == 0).sum()}")

# Cachear tokens una sola vez por archivo (338 archivos, no 56k lecturas)
print("\nPreprocesando y cacheando archivos...")
all_files = set(df_labeled["file1"]) | set(df_labeled["file2"])
file_tokens_cache = {}
for fname in tqdm(all_files, desc="Tokenizando archivos"):
    code = normalize_code(read_code(fname))
    file_tokens_cache[fname] = tokenize(code)

def compute_features_cached(f1, f2, emb_cos):
    tokens1 = file_tokens_cache.get(f1, [])
    tokens2 = file_tokens_cache.get(f2, [])

    # Primeros 100 tokens como string para Levenshtein/Jaro (rendimiento)
    t1 = " ".join(tokens1[:100])
    t2 = " ".join(tokens2[:100])

    lev_dist = RapidLev.distance(t1, t2)
    max_len  = max(len(t1), len(t2))
    lev_ratio = 1.0 - (lev_dist / max_len) if max_len > 0 else 1.0

    return {
        "code1_id":          f1,
        "code2_id":          f2,
        "jaccard_tokens":    jaccard(tokens1, tokens2),
        "jaccard_bigrams":   jaccard(get_ngrams(tokens1, 2), get_ngrams(tokens2, 2)),
        "jaccard_trigrams":  jaccard(get_ngrams(tokens1, 3), get_ngrams(tokens2, 3)),
        "dice":              dice(tokens1, tokens2),
        "levenshtein_dist":  lev_dist,
        "levenshtein_ratio": lev_ratio,
        "jaro":              Jaro.similarity(t1, t2),
        "jaro_winkler":      JaroWinkler.similarity(t1, t2),
        "cosine_similarity": cosine_bow(tokens1, tokens2),
        "embedding_cosine":  round(float(emb_cos), 6),
    }

# Calcular features para TODOS los pares
print("\nCalculando features para todos los pares...")
rows = []
for _, row in tqdm(df_labeled.iterrows(), total=len(df_labeled), desc="Procesando pares"):
    record = compute_features_cached(row["file1"], row["file2"], row["similarity"])
    record["label"] = row["plagiarism"]
    rows.append(record)

# Columnas en orden: IDs → label → features
COLS = [
    "code1_id", "code2_id", "label",
    "jaccard_tokens", "jaccard_bigrams", "jaccard_trigrams",
    "dice", "levenshtein_dist", "levenshtein_ratio",
    "jaro", "jaro_winkler", "cosine_similarity", "embedding_cosine"
]
dataset = pd.DataFrame(rows)[COLS]

CSV_PATH = "plagiarism_dataset.csv"
dataset.to_csv(CSV_PATH, index=False)

print(f"\nDataset guardado: {CSV_PATH}")
print(f"  Filas: {len(dataset)} | Columnas: {len(dataset.columns)}")
print(f"  Distribución de labels:\n{dataset['label'].value_counts().to_string()}")
display(dataset.head(10))


## Modelo de Detección de Plagio

Entrenamos un Random Forest con las 10 features de similitud y evaluamos con métricas estándar de clasificación.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

# Leer el dataset (por si se corre esta celda independientemente)
dataset = pd.read_csv("plagiarism_dataset.csv")

FEATURE_COLS = [
    "jaccard_tokens", "jaccard_bigrams", "jaccard_trigrams",
    "dice", "levenshtein_dist", "levenshtein_ratio",
    "jaro", "jaro_winkler", "cosine_similarity", "embedding_cosine"
]

X = dataset[FEATURE_COLS].values
y = dataset["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="f1", n_jobs=-1)
print(f"F1 CV (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No Plagio", "Plagio"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["No Plagio", "Plagio"]).plot(
    ax=axes[0], cmap="RdPu", colorbar=False
)
axes[0].set_title("Matriz de Confusión")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#c77dff", lw=2,
             label=f"AUC = {roc_auc_score(y_test, y_proba):.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Curva ROC")
axes[1].legend()

importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
importances.plot(kind="barh", ax=axes[2], color="#c77dff")
axes[2].set_title("Importancia de Features")
axes[2].set_xlabel("Importancia")

plt.tight_layout()
plt.show()
